# 13 · ML Systems Overview — The Model Is the Small Part

**Part 2 · Machine Learning Systems** · *Prerequisites: 00–01 (Part 1 recommended)* ·
*Book mapping: DMLS ch. 1–2*

Part 1 built systems that compute what you told them to. An **ML system** computes a
function *learned from data* — and everything around that learned function is still a data
system: pipelines feed it (nb 10–11), stores serve it (nb 03), its predictions are derived
data (nb 12), and its failures are often data failures wearing a model costume. The model
is usually <10% of the code and ~100% of the mystique.

**Research ML vs production ML** — the gap this whole part is about:

| | Research | Production |
|---|---|---|
| Objective | beat the benchmark metric | move a business metric |
| Data | clean, static, given | messy, shifting (nb 19), *yours to build* (nb 14–15) |
| Compute priority | fast **training** | fast, cheap **inference** (nb 18) |
| Requirements | good average performance | also: latency p99, fairness, explainability, cost |
| Done when | paper accepted | never — it loops (retraining, nb 20) |

**Learning objectives.**

1. Decide when ML is (and isn't) the right tool.
2. **Frame** one product problem three ways and show the framing — not the model — changes
   who gets acted on.
3. Show that the **label is a design choice** with measurable consequences.
4. Decouple multiple objectives so business trade-offs don't require retraining.
5. Turn a classifier into a **decision policy** with cost-sensitive thresholds — and watch
   "accuracy" stop being the point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=13)

## 1. When (not) to use ML

ML earns its complexity when **all** of these hold: patterns exist (and are complex enough
that hand-written rules rot), data about them is available (or collectable — nb 15),
wrong predictions are tolerable at some rate, the decision repeats at scale (unit economics
of a model beat human judgment), and the patterns *change* (rules would need constant
manual updates; a retraining pipeline handles it — nb 20).

ML is the wrong tool when: you need guarantees or auditable logic (tax calculation), one
correct answer exists and is computable (use the formula), there's no data and no way to
get it, errors are catastrophic and irreversible, or — the most common case — **a simple
heuristic captures 90% of the value**. Rule one of ML systems, straight from the
practitioners: *start without ML; make the baseline earn its replacement* (nb 17
operationalizes this as the baseline ladder).

## 2. One problem, three framings

Product ask: *"Reduce churn — we can afford to send retention offers to 500 users this
month."* Nothing in that sentence says "binary classifier". Framing is the highest-leverage
modeling decision, made before any model exists. We synthesize a subscription business
whose ground truth we control, so every claim below is checkable:

In [ ]:
N = 8_000
users = pd.DataFrame({
    "tenure_months": rng.gamma(4, 6, N).clip(1, 72),
    "weekly_active_days": rng.uniform(0, 7, N),
    "support_tickets": rng.poisson(1.0, N),
    "monthly_spend": np.round(rng.gamma(2, 15, N) + 5, 2),   # per-user value differs!
})

# Hidden truth: risk rises with tickets & inactivity, falls with tenure.
risk = (0.9 * users.support_tickets - 0.55 * users.weekly_active_days
        - 0.05 * users.tenure_months + 1.2)
p_churn = 1 / (1 + np.exp(-risk))
# Ground-truth days until each user would churn (small = urgent):
days_to_churn = rng.exponential(45 / p_churn.clip(0.02))

users["churn_30d"] = (days_to_churn <= 30).astype(int)       # THE label... one of them
print(f"{N} users, 30-day churn rate: {users.churn_30d.mean():.1%}")

features = ["tenure_months", "weekly_active_days", "support_tickets", "monthly_spend"]
train, test = train_test_split(users.assign(days=days_to_churn),
                               test_size=0.5, random_state=0)

In [ ]:
BUDGET = 500

# Framing A - binary classification: "who is most LIKELY to churn in 30 days?"
clf = LogisticRegression(max_iter=1000).fit(train[features], train.churn_30d)
p_hat = clf.predict_proba(test[features])[:, 1]
pick_probability = set(test.index[np.argsort(-p_hat)[:BUDGET]])

# Framing B - regression: "who churns SOONEST?" (urgency, not likelihood)
reg = LinearRegression().fit(train[features], train.days)
days_hat = reg.predict(test[features])
pick_urgency = set(test.index[np.argsort(days_hat)[:BUDGET]])

# Framing C - value ranking: "where is the most REVENUE at risk?"
value_at_risk = p_hat * test.monthly_spend.to_numpy()
pick_value = set(test.index[np.argsort(-value_at_risk)[:BUDGET]])

overlap_ab = len(pick_probability & pick_urgency) / BUDGET
overlap_ac = len(pick_probability & pick_value) / BUDGET
print(f"overlap of the 500 chosen users - likelihood vs urgency: {overlap_ab:.0%}, "
      f"likelihood vs value-at-risk: {overlap_ac:.0%}")
assert 0.2 < overlap_ac < 0.95, \
    "different framings act on substantially different PEOPLE - same data, same budget"
assert pick_probability != pick_value != pick_urgency

Same dataset, same budget, three defensible framings — and hundreds of *different humans*
get the offer. No metric inside any one framing can tell you which is right; only the
product goal can ("save most subscribers" → A; "save most revenue" → C; "we need 2 weeks
of lead time to intervene" → B). **Framing is a product decision expressed in math.**

### The label is a design choice too

"Churn" isn't a fact in the data; it's a definition you wrote. Move the window from 30 to
60 days and the *ground truth itself* changes for a measurable slice of users:

In [ ]:
churn_60d = (test.days <= 60)
flipped = (churn_60d != (test.days <= 30)).mean()
print(f"users whose LABEL flips between 30d and 60d definitions: {flipped:.1%}")
assert flipped > 0.05, "the 'truth' the model learns is downstream of a design meeting"

Every label carries such choices (what counts as "fraud"? which click is a "positive"?),
and they propagate into everything trained on them. Write label definitions down like API
contracts — nb 15 goes deep on where labels come from at all.

## 3. Multiple objectives: decouple, don't blend

Real systems optimize several things at once — a feed wants engagement *and* content
quality; ranking wants relevance *and* revenue. The tempting shortcut: blend them into one
training label (`0.7·clicks + 0.3·quality`) and train one model. The trap: **every
business re-weighting now requires retraining** (and re-validating, and re-deploying —
nb 20's whole pipeline).

The decoupled design: one model (or score) per objective, combined **at decision time**:

In [ ]:
items = pd.DataFrame({
    "engagement_score": rng.beta(2, 5, 300),      # from model 1 (clicks)
    "quality_score": rng.beta(5, 2, 300),         # from model 2 / editorial score
})

def rank_feed(alpha: float, k: int = 10) -> list[int]:
    final = alpha * items.engagement_score + (1 - alpha) * items.quality_score
    return list(np.argsort(-final)[:k])

engagement_heavy = rank_feed(alpha=0.9)
balanced = rank_feed(alpha=0.5)
print(f"top-10 (alpha=0.9): {engagement_heavy}\ntop-10 (alpha=0.5): {balanced}")
assert engagement_heavy != balanced, \
    "the trade-off moved with a CONFIG CHANGE - zero retraining, instant rollback"
assert set(engagement_heavy) & set(balanced), "sane overlap: same underlying scores"

The pattern generalizes (DMLS calls it *decoupling objectives*): objectives that evolve at
different speeds — or are owned by different teams — deserve separate models joined by an
explicit, tunable policy. The policy layer is also where guardrails live (filters, floors,
diversity constraints), visible and versioned instead of baked into weights.

## 4. From probabilities to decisions: cost-sensitive thresholds

A classifier outputs probabilities; the *business* executes decisions. The bridge is a
threshold — and `0.5` is almost never the right one, because errors are not symmetric.
Our retention economics: an offer costs **$10** to send; a churner who gets the offer is
saved with probability **0.35**, preserving **$100** of lifetime value. So intervening on a
user with churn probability `p` has expected profit `p × 0.35 × $100 − $10` — positive
exactly when `p > 10/35 ≈ 0.29`. Let's watch the money agree with the algebra:

In [ ]:
OFFER_COST, SAVE_RATE, LTV = 10.0, 0.35, 100.0

def expected_profit(threshold: float) -> float:
    """Send offers to everyone with p_hat >= threshold; tally expected profit
    against the TEST set's actual churn outcomes."""
    chosen = p_hat >= threshold
    churners_reached = (test.churn_30d.to_numpy() == 1) & chosen
    return float(churners_reached.sum() * SAVE_RATE * LTV - chosen.sum() * OFFER_COST)

thresholds = np.linspace(0.05, 0.9, 60)
profits = np.array([expected_profit(t) for t in thresholds])
best_t = float(thresholds[profits.argmax()])

fig, ax = plt.subplots()
ax.plot(thresholds, profits)
ax.axvline(0.5, color="#9e9e9e", linestyle=":", label="default 0.5")
ax.axvline(10 / 35, color="#c62828", linestyle="--", label="break-even p = cost/save")
ax.set_xlabel("intervention threshold on p(churn)")
ax.set_ylabel("expected profit ($)")
ax.set_title("The threshold is a business dial, not a math constant")
ax.legend()
plt.show()

print(f"best threshold ~{best_t:.2f}; profit at best: ${profits.max():,.0f} "
      f"vs at 0.5: ${expected_profit(0.5):,.0f}")
assert abs(best_t - 10 / 35) < 0.12, "the profit-optimal threshold sits near cost/save"
assert profits.max() > expected_profit(0.5) * 1.1, \
    "the default threshold left >10% of the money on the table"

Generalization worth memorizing: with false-positive cost $c_{fp}$ and false-negative cost
$c_{fn}$, act when $p > c_{fp} / (c_{fp} + c_{fn})$. The model didn't change; the
*decision policy* did — and notice it can keep changing (offer cost drops, LTV rises)
**without retraining**, because we deployed a probability + a policy, not a hard-coded
verdict. Same decoupling lesson as §3, now with dollars attached.

## 5. The lifecycle — and the map of Part 2

An ML system is a **loop**, not a pipeline: scope the problem (today) → engineer data
(nb 14) → build training data (nb 15) → engineer features (nb 16) → train & evaluate
(nb 17) → deploy & serve (nb 18) → monitor in the world (nb 19) → the world shifts →
retrain and test in production (nb 20) → repeat forever, on infrastructure that makes the
loop cheap (nb 21). Every arrow in that loop is a Part 1 concept doing a tour of duty:
Parquet and logs feed it, derived-data thinking versions it, and the "rebuild from
history" superpower (nb 12) becomes "retrain from the feature log".

## 6. Exercises

1. **Frame it.** "Detect toxic comments" — give a binary framing, a multi-label framing
   (severity/type), and a ranking framing (human review queue). For each: what's the
   label, what does a false positive cost, and which team's workflow does it change?
2. **ML or not?** (a) Compute shipping cost from weight and zone; (b) flag fraudulent
   card payments; (c) a startup's first spam filter with 200 emails collected; (d) predict
   tomorrow's staffing for a call center with 3 years of hourly volume data. Verdict +
   one-line reason each.
3. **Ads in the feed.** Why do mature feed systems train *separate* models for organic
   engagement and ad click-through, then combine with a tunable auction/policy layer —
   instead of one model over both? Give two decoupling reasons and one guardrail the
   policy layer enforces.
4. **Derive the dial.** From a cost matrix (act/not-act × churns/doesn't), derive the
   break-even threshold formula and verify it equals `cost/(save_rate × LTV)` for our
   numbers. What changes if the offer itself annoys loyal users (adds a cost to true
   negatives acted on)?
5. **Two kinds of metrics.** Offline, framing A is judged by ROC-AUC; the business
   tracks monthly retention. Name three distinct reasons the first can improve while the
   second doesn't move (foreshadowing: nb 17's slices, nb 19's shifts, nb 20's
   experiments).

### Solutions

**1.** Binary: label toxic/not per comment; FP = silenced legitimate user (trust damage);
changes *automated moderation*. Multi-label: {insult, threat, spam, ...} with per-class
thresholds; FP costs differ per class (a wrong "threat" flag is worse); changes *policy
enforcement granularity*. Ranking: order comments by p(toxic) for human review under a
fixed reviewer budget (our BUDGET=500, verbatim); FP = wasted reviewer minutes only —
which is why ranking-to-humans is the gentlest first deployment of a risky model.

**2.** (a) No ML — a formula exists; use it. (b) ML — adversarial, shifting patterns at
scale (and nb 25's capstone). (c) Not yet — 200 examples: ship a rule/blocklist baseline
and *start logging* (nb 15's collection loop). (d) ML (or even classical forecasting —
which *is* the baseline to beat): repeating decision, seasonal patterns, tolerable errors.

**3.** Decoupling: (i) the trade-off (revenue vs engagement) is a *policy* the business
retunes weekly — §3's config change, not a retrain; (ii) the objectives drift at
different rates and are owned by different teams with different labels and cadences.
Guardrail example: ad load caps / quality floors enforced in the combiner where they are
visible and auditable, not implicit in weights.

**4.** Act iff `p·save_rate·LTV − offer_cost > 0` ⇒ `p* = offer_cost/(save_rate·LTV)` =
10/35 ≈ 0.29 ✓ (the assert in §4 measured exactly this). If offers annoy non-churners,
add `−(1−p)·annoy_cost` to the act branch: `p* = (offer_cost + annoy_cost)/(save_rate·LTV
+ annoy_cost)` — the bar rises, the model is untouched, the *policy* absorbs the new
economics. That's why the policy must live outside the weights.

**5.** (i) **Slice trouble** (nb 17): AUC averages over users; the gain concentrated on
users you'd never act on (below every sane threshold). (ii) **Shift** (nb 19): offline
test data is last quarter; the deployed world moved. (iii) **The action, not the
prediction, makes money** (nb 20): better ranking of who churns ≠ better *saving* —
offer efficacy, cannibalization, and feedback loops live outside the model; only an
online experiment measures them.

## 7. Takeaways

- Production ML is a data system with a learned component: inference-first, requirement-
  laden, never "done". Start from the heuristic baseline and make ML earn its keep.
- **Framing decides who gets acted on** — likelihood vs urgency vs value are different
  products. The **label is designed**, not found; write it down like an API.
- **Decouple objectives**: separate scores, explicit combination policy. Business dials
  (α, thresholds) belong in config, not in weights — retuning must not mean retraining.
- Classifiers output probabilities; businesses execute policies. `p* = c_fp/(c_fp+c_fn)`
  — thresholds are economics, and 0.5 is a superstition.
- The lifecycle is a loop; Part 2 walks it one notebook per stage.

**Further reading.** DMLS ch. 1–2; Sculley et al., *Hidden Technical Debt in Machine
Learning Systems* (NeurIPS 2015) — the "model is the small part" paper; Google's *Rules
of ML* (rule #1: don't be afraid to launch without ML).

**Next:** [14 · Data engineering for ML](14-data-engineering-for-ml.ipynb) — the loop's
first stage: getting event data into shapes models can eat, at speed.